# M5 — Route-assign  (Christian: summary-only, time split, review gate)

Based on **Christian**'s routing model: **summary only** (word 1–2-grams +
character 3–5-grams), class-balanced logistic regression, on a **chronological
split**. Channel / plan / tenure are deliberately excluded (their link to the
team drifts month to month). `operations` is merged into `support` (same team,
two names). A top suggestion below `REVIEW_THRESHOLD` (0.5) is marked
*needs human review*. Saves `models/m5_route.joblib`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import northstar_prep as P
from sklearn.metrics import f1_score, accuracy_score

df = P.load_clean(merge_team_aliases=True)         # operations -> support
FEATS, TARGET = ["case_summary"], "assigned_team"
REVIEW_THRESHOLD = 0.5
tr, va, te = P.time_split(df)
pipe = P.make_clf([P.t_word(), P.t_char()]).fit(tr[FEATS], tr[TARGET])
proba = pipe.predict_proba(te[FEATS]); pred = pipe.classes_[proba.argmax(1)]
conf = proba.max(1)
metrics = {"macro_F1": round(f1_score(te[TARGET], pred, average="macro"), 3),
           "accuracy": round(accuracy_score(te[TARGET], pred), 3),
           "baseline_acc": round(df[TARGET].value_counts(normalize=True).iloc[0], 3),
           "pct_needs_review": round((conf < REVIEW_THRESHOLD).mean(), 3)}
print(metrics)

In [ ]:
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["predicted"]  = pred
test["confidence"] = conf.round(3)

full = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATS], df[TARGET])
path = P.save_bundle("m5_route", {
    "model": full, "target": TARGET, "features": FEATS, "classes": sorted(df[TARGET].unique()),
    "kind": "multiclass", "label": "Suggested route", "review_threshold": REVIEW_THRESHOLD,
    "metrics": metrics, "test": test,
    "meta": {"author":"Christian","split":"time","inputs":"summary only",
             "note":"operations merged into support; review gate at 0.5"}})
print("saved", path, "| test rows:", len(test))